# BORG Galaxy spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "borg_blue_monster_spectrum.gif"

FPS = 24
DURATION_SEC = 8
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#030711"
GRID = "#16324a"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
BLUE = "#0b63ff"
ORANGE = "#ff6b32"
PURPLE = "#b76cff"
GREEN = "#48ffb3"
WHITE = "#ffffff"

# =========================
# Data: illustrative rest-frame spectrum
# =========================

wl = np.linspace(900, 7600, 2200)  # Rest-frame Angstrom

def gaussian(x, mu, amp, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

# Very blue UV continuum: young, dust-poor stellar population
# F_lambda decreases with wavelength.
continuum = 0.34 + 0.95 * (wl / 1500) ** -1.55

# Lyman break / IGM absorption edge, shown schematically
break_factor = np.ones_like(wl)
break_factor[wl < 1216] = 0.10 + 0.08 * (wl[wl < 1216] - 900) / (1216 - 900)

spectrum = continuum * break_factor

# Strong nebular emission lines, schematic
lines = [
    (1216, 0.65, 22, "Lyα"),
    (1549, 0.22, 26, "C IV"),
    (1909, 0.20, 30, "C III]"),
    (3727, 0.38, 42, "[O II]"),
    (4861, 0.28, 38, "Hβ"),
    (5007, 0.95, 44, "[O III]"),
    (6563, 0.55, 55, "Hα"),
]

for mu, amp, sigma, _ in lines:
    spectrum += gaussian(wl, mu, amp, sigma)

# Small deterministic texture: keeps it looking like observed spectrum
rng = np.random.default_rng(7)
noise = rng.normal(0, 0.012, size=wl.size)
spectrum_noisy = spectrum + noise

# Normalize to plotting range
spectrum_noisy = spectrum_noisy / np.nanmax(spectrum_noisy) * 1.18

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.set_xlim(900, 7600)
ax.set_ylim(0.0, 1.45)

for spine in ax.spines.values():
    spine.set_color("#95a3b5")
    spine.set_linewidth(1.2)

ax.tick_params(colors=TEXT, labelsize=12, length=6)

ax.set_xticks([1000, 1500, 2000, 3000, 4000, 5000, 6000, 7000])
ax.set_xticklabels(["1000", "1500", "2000", "3000", "4000", "5000", "6000", "7000"])

ax.set_yticks([0.25, 0.50, 0.75, 1.00, 1.25])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00", "1.25"])

ax.grid(True, which="major", color=GRID, linestyle=":", linewidth=1.1, alpha=0.75)

ax.set_xlabel(r"Rest-frame wavelength  $\lambda$  (Å)", color=TEXT, fontsize=18, labelpad=14)
ax.set_ylabel("Relative flux", color=TEXT, fontsize=18, labelpad=12)

fig.text(
    0.5, 0.94,
    "BoRG / JWST BLUE MONSTER — ILLUSTRATIVE SPECTRUM",
    ha="center",
    va="center",
    color="#d7dde8",
    fontsize=23,
    fontweight="bold"
)

fig.text(
    0.5, 0.90,
    "young dust-poor galaxy • blue UV continuum • strong nebular lines",
    ha="center",
    va="center",
    color=MUTED,
    fontsize=14,
    fontweight="bold"
)

# =========================
# Static annotations
# =========================

# Lyman break region
ax.axvspan(900, 1216, color=BLUE, alpha=0.08)
ax.axvline(1216, color=PURPLE, alpha=0.35, linewidth=1.4)

ax.text(
    1060, 1.27,
    "LYMAN\nBREAK",
    color=PURPLE,
    fontsize=13,
    fontweight="bold",
    ha="center"
)

# Continuum note
ax.text(
    2350, 1.18,
    "BLUE UV CONTINUUM",
    color=CYAN,
    fontsize=16,
    fontweight="bold",
    ha="center"
)

ax.text(
    5300, 1.18,
    "STRONG NEBULAR\nEMISSION LINES",
    color=ORANGE,
    fontsize=16,
    fontweight="bold",
    ha="center"
)

# Band labels
bands = [
    (1350, "FAR-UV"),
    (2200, "UV"),
    (3800, "NEAR-UV / BLUE"),
    (5400, "OPTICAL"),
    (6850, "RED / Hα"),
]

for bx, label in bands:
    col = CYAN if bx < 4000 else ORANGE
    ax.text(
        bx,
        0.055,
        label,
        color=col,
        fontsize=11,
        fontweight="bold",
        ha="center"
    )

# Emission line guide marks
for mu, amp, sigma, label in lines:
    if mu < 900 or mu > 7600:
        continue

    color = PURPLE if mu <= 1909 else ORANGE

    ax.axvline(mu, color=color, alpha=0.16, linewidth=1.0)

    y_label = 1.35 if label not in ("[O III]", "Hα") else 1.30

    ax.text(
        mu,
        y_label,
        label,
        color=color,
        fontsize=11,
        ha="center",
        va="top",
        fontweight="bold",
        rotation=0
    )

ax.text(
    0.98, 0.05,
    "SCHEMATIC — NOT OBSERVED DATA",
    transform=ax.transAxes,
    color="#415064",
    fontsize=9,
    ha="right"
)

ax.text(
    0.50, -0.22,
    "rest-frame illustration: observed JWST wavelengths would be redshifted by (1 + z)",
    transform=ax.transAxes,
    color=TEXT,
    fontsize=11,
    ha="center",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358"
    )
)

# =========================
# Animated artists
# =========================

fill_poly = ax.fill_between([], [], [], color=BLUE, alpha=0.0)

glow1, = ax.plot([], [], color=CYAN, linewidth=9, alpha=0.10)
glow2, = ax.plot([], [], color=CYAN, linewidth=5, alpha=0.18)
main_line, = ax.plot([], [], color=CYAN, linewidth=2.6)

line_glow1, = ax.plot([], [], color=ORANGE, linewidth=9, alpha=0.09)
line_glow2, = ax.plot([], [], color=ORANGE, linewidth=5, alpha=0.16)
line_overlay, = ax.plot([], [], color=ORANGE, linewidth=2.2, alpha=0.0)

cursor, = ax.plot([], [], marker="o", markersize=6, color=WHITE, alpha=0.95)

# =========================
# Animation
# =========================

def ease(t):
    return 1 - (1 - t) ** 3


def update(frame):
    global fill_poly

    t = ease(frame / (FRAMES - 1))
    xmax = wl.min() + t * (wl.max() - wl.min())

    visible = wl <= xmax

    xv = wl[visible]
    yv = spectrum_noisy[visible]

    main_line.set_data(xv, yv)
    glow1.set_data(xv, yv)
    glow2.set_data(xv, yv)

    if len(xv) > 0:
        cursor.set_data([xv[-1]], [yv[-1]])

    fill_poly.remove()
    fill_poly = ax.fill_between(
        xv,
        0,
        yv,
        color=BLUE,
        alpha=0.12
    )

    # Orange overlay only near strong optical nebular lines
    line_region = visible & (
        ((wl > 3650) & (wl < 3820)) |
        ((wl > 4800) & (wl < 5080)) |
        ((wl > 6480) & (wl < 6650))
    )

    x_lines = wl[line_region]
    y_lines = spectrum_noisy[line_region]

    line_overlay.set_data(x_lines, y_lines)
    line_glow1.set_data(x_lines, y_lines)
    line_glow2.set_data(x_lines, y_lines)

    line_overlay.set_alpha(0.95 if xmax > 3600 else 0.0)

    return (
        main_line, glow1, glow2,
        line_overlay, line_glow1, line_glow2,
        cursor, fill_poly
    )


anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

# BORG FOV Animation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Ellipse, Circle
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("Infographics/Telemetry/media-site/animations/blue_monsters")
OUT_DIR.mkdir(exist_ok=True)

WEBM_PATH = OUT_DIR / "blue_monsters_deep_field_layer.webm"

FPS = 24
DURATION_SEC = 12
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#020510"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
DEEP_BLUE = "#0b63ff"
ORANGE = "#ff9a3c"
RED = "#ff5a45"
WHITE = "#f0f6ff"
PURPLE = "#b76cff"

rng = np.random.default_rng(42)

# =========================
# Scene geometry
# =========================

W, H = 16, 9
N_GALAXIES = 180
N_TARGETS = 12
N_STARS = 260

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

# Убираем все отступы matplotlib
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_position([0, 0, 1, 1])

# Сжимаем "камеру"
PAD = 0.25  # поиграйся: 0.15–0.4

ax.set_xlim(PAD, W - PAD)
ax.set_ylim(PAD, H - PAD)

ax.set_aspect("equal")
ax.axis("off")

# =========================
# Star background
# =========================

star_x = rng.uniform(0, W, N_STARS)
star_y = rng.uniform(0, H, N_STARS)
star_s = rng.uniform(1, 8, N_STARS)
star_alpha = rng.uniform(0.15, 0.65, N_STARS)

ax.scatter(
    star_x,
    star_y,
    s=star_s,
    c=WHITE,
    alpha=star_alpha,
    linewidths=0
)

# =========================
# Background galaxies
# =========================

galaxies = []

palette = [
    "#f7d9a0", "#d7c7ff", "#ffb27a", "#dce7ff",
    "#b7c7ff", "#ffd0c2", "#ff8f70", "#b7e8ff"
]

for _ in range(N_GALAXIES):
    x = rng.uniform(0.4, W - 0.4)
    y = rng.uniform(0.4, H - 0.4)

    width = rng.uniform(0.06, 0.32)
    height = width * rng.uniform(0.28, 0.85)
    angle = rng.uniform(0, 180)

    color = rng.choice(palette)
    alpha = rng.uniform(0.22, 0.72)

    galaxies.append((x, y, width, height, angle, color, alpha))

# Draw glow + body for galaxies
for x, y, width, height, angle, color, alpha in galaxies:
    glow = Ellipse(
        (x, y),
        width * 2.4,
        height * 2.4,
        angle=angle,
        facecolor=color,
        edgecolor="none",
        alpha=alpha * 0.10
    )
    body = Ellipse(
        (x, y),
        width,
        height,
        angle=angle,
        facecolor=color,
        edgecolor="none",
        alpha=alpha
    )
    core = Ellipse(
        (x, y),
        width * 0.34,
        height * 0.34,
        angle=angle,
        facecolor=WHITE,
        edgecolor="none",
        alpha=min(alpha + 0.15, 0.85)
    )

    ax.add_patch(glow)
    ax.add_patch(body)
    ax.add_patch(core)

# =========================
# Blue monster target layer
# =========================

target_positions = []

# choose positions with some spacing
attempts = 0
while len(target_positions) < N_TARGETS and attempts < 2000:
    attempts += 1
    x = rng.uniform(1.0, W - 1.0)
    y = rng.uniform(1.0, H - 1.0)

    if all((x - px) ** 2 + (y - py) ** 2 > 1.0 for px, py in target_positions):
        target_positions.append((x, y))

target_artists = []
ring_artists = []
label_artists = []
connector_artists = []

for i, (x, y) in enumerate(target_positions, start=1):
    size = rng.uniform(0.13, 0.23)
    ratio = rng.uniform(0.55, 0.9)
    angle = rng.uniform(0, 180)

    # target glow
    glow = Ellipse(
        (x, y),
        size * 3.8,
        size * 3.8 * ratio,
        angle=angle,
        facecolor=CYAN,
        edgecolor="none",
        alpha=0.0,
        zorder=20
    )

    # target galaxy
    body = Ellipse(
        (x, y),
        size,
        size * ratio,
        angle=angle,
        facecolor=CYAN,
        edgecolor=WHITE,
        linewidth=0.3,
        alpha=0.0,
        zorder=21
    )

    core = Ellipse(
        (x, y),
        size * 0.36,
        size * 0.36,
        angle=angle,
        facecolor=WHITE,
        edgecolor="none",
        alpha=0.0,
        zorder=22
    )

    ring1 = Circle(
        (x, y),
        radius=size * 1.65,
        edgecolor=CYAN,
        facecolor="none",
        linewidth=1.2,
        alpha=0.0,
        zorder=23
    )

    ring2 = Circle(
        (x, y),
        radius=size * 2.45,
        edgecolor=DEEP_BLUE,
        facecolor="none",
        linewidth=0.9,
        alpha=0.0,
        zorder=23
    )

    # labels: offset alternates to avoid clutter
    dx = 0.45 if i % 2 else -0.45
    dy = 0.32 if i % 3 else -0.36
    ha = "left" if dx > 0 else "right"

    label = ax.text(
        x + dx,
        y + dy,
        f"BM-{i:02d}",
        color=CYAN,
        fontsize=8.5,
        fontweight="bold",
        ha=ha,
        va="center",
        alpha=0.0,
        zorder=24
    )

    connector, = ax.plot(
        [x, x + dx * 0.78],
        [y, y + dy * 0.78],
        color=CYAN,
        linewidth=0.7,
        alpha=0.0,
        zorder=23
    )

    for artist in [glow, body, core, ring1, ring2]:
        ax.add_patch(artist)

    target_artists.append((glow, body, core))
    ring_artists.append((ring1, ring2))
    label_artists.append(label)
    connector_artists.append(connector)

# =========================
# HUD overlay
# =========================

title = ax.text(
    0.5,
    8.55,
    "BoRG / HST DEEP FIELD — BLUE MONSTER CANDIDATE FILTER",
    color=TEXT,
    fontsize=18,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=40
)

subtitle = ax.text(
    0.5,
    8.22,
    "layer OFF",
    color=MUTED,
    fontsize=12,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=40
)

status_box = ax.text(
    15.45,
    8.45,
    "BLUE LAYER: OFF",
    color=MUTED,
    fontsize=11,
    fontweight="bold",
    ha="right",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358",
        alpha=0.85
    ),
    zorder=40
)

counter = ax.text(
    15.45,
    0.42,
    "0 candidates highlighted",
    color=MUTED,
    fontsize=10,
    ha="right",
    va="center",
    zorder=40
)

# subtle frame
frame_lines = []
frame_style = dict(color="#2b4358", linewidth=1.0, alpha=0.45, zorder=35)
frame_lines.append(ax.plot([0.35, 3.0], [8.75, 8.75], **frame_style)[0])
frame_lines.append(ax.plot([0.35, 0.35], [8.75, 7.95], **frame_style)[0])
frame_lines.append(ax.plot([13.0, 15.65], [8.75, 8.75], **frame_style)[0])
frame_lines.append(ax.plot([15.65, 15.65], [8.75, 7.95], **frame_style)[0])
frame_lines.append(ax.plot([0.35, 2.1], [0.25, 0.25], **frame_style)[0])
frame_lines.append(ax.plot([0.35, 0.35], [0.25, 0.95], **frame_style)[0])
frame_lines.append(ax.plot([13.9, 15.65], [0.25, 0.25], **frame_style)[0])
frame_lines.append(ax.plot([15.65, 15.65], [0.25, 0.95], **frame_style)[0])

# =========================
# Animation helpers
# =========================

def smoothstep(edge0, edge1, x):
    if edge0 == edge1:
        return 1.0 if x >= edge1 else 0.0
    t = np.clip((x - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def layer_alpha(t):
    """
    Smooth loop:
    0.00-0.20: layer off
    0.20-0.40: fade in
    0.40-0.70: layer on
    0.70-0.90: fade out
    0.90-1.00: layer off

    Good for seamless GIF loop.
    """
    fade_in = smoothstep(0.15, 0.32, t)
    fade_out = 1.0 - smoothstep(0.74, 0.92, t)

    return fade_in * fade_out

# =========================
# Animation update
# =========================

def update(frame):
    t = frame / (FRAMES - 1)
    a = layer_alpha(t)

    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * (t * 6))
    ring_scale = 1.0 + 0.28 * pulse

    for idx, ((glow, body, core), (ring1, ring2), label, connector) in enumerate(
        zip(target_artists, ring_artists, label_artists, connector_artists)
    ):
        delay = idx * 0.012
        local_a = a * smoothstep(0.0, 0.12, max(t - delay, 0))

        glow.set_alpha(0.22 * local_a)
        body.set_alpha(0.92 * local_a)
        core.set_alpha(0.98 * local_a)

        base_r1 = 0.20
        base_r2 = 0.31

        ring1.set_radius(base_r1 * ring_scale)
        ring2.set_radius(base_r2 * (1.0 + 0.38 * pulse))

        ring1.set_alpha((0.55 + 0.30 * pulse) * local_a)
        ring2.set_alpha((0.30 + 0.28 * (1 - pulse)) * local_a)

        label_alpha = local_a * smoothstep(0.42, 0.55, t)
        label.set_alpha(label_alpha)
        connector.set_alpha(label_alpha * 0.75)

    if a > 0.08:
        subtitle.set_text("compact blue systems isolated by color-selection layer")
        subtitle.set_color(CYAN)
        status_box.set_text("BLUE LAYER: ON")
        status_box.set_color(CYAN)
        counter.set_text(f"{N_TARGETS} candidates highlighted")
        counter.set_color(CYAN)
    else:
        subtitle.set_text("layer OFF")
        subtitle.set_color(MUTED)
        status_box.set_text("BLUE LAYER: OFF")
        status_box.set_color(MUTED)
        counter.set_text("0 candidates highlighted")
        counter.set_color(MUTED)

    return (
        [subtitle, status_box, counter]
        + [artist for group in target_artists for artist in group]
        + [artist for group in ring_artists for artist in group]
        + label_artists
        + connector_artists
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = FFMpegWriter(fps=FPS, codec="libvpx-vp9", extra_args=["-crf", "32", "-b:v", "0", "-pix_fmt", "yuv420p"])
anim.save(WEBM_PATH, writer=writer)

plt.close(fig)

print(f"Saved: {WEBM_PATH}")
print(f"Saved: {WEBM_PATH}")


# Diagram from the work

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "blue_monsters_beta_muv_dark.gif"

FPS = 24
DURATION_SEC = 10
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#030711"
GRID = "#16324a"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
BLUE = "#4d8cff"
MAGENTA = "#ff23ff"
ORANGE = "#ff9a66"
WHITE = "#f0f6ff"
GRAY = "#9aa7b8"

rng = np.random.default_rng(17)

# =========================
# Synthetic illustrative data
# =========================

# Approximate layout inspired by the reference figure, not real data.
borg = np.array([
    [-22.35, -2.02, 0.06, 0.10],
    [-22.05, -2.41, 0.07, 0.07],
    [-21.55, -2.21, 0.15, 0.06],
    [-21.45, -2.30, 0.18, 0.07],
    [-21.42, -2.51, 0.16, 0.16],
    [-21.28, -2.21, 0.17, 0.07],
    [-21.10, -2.15, 0.06, 0.09],
    [-20.75, -2.42, 0.13, 0.09],
    [-20.68, -2.37, 0.22, 0.36],
    [-20.65, -2.05, 0.19, 0.13],
    [-20.62, -2.16, 0.14, 0.15],
    [-20.45, -2.23, 0.42, 0.26],
    [-21.58, -1.63, 0.14, 0.06],
])

blue_monsters = np.array([
    [-21.50, -2.36, 0.08, 0.10],
    [-21.47, -2.52, 0.10, 0.05],
    [-21.19, -2.46, 0.20, 0.08],
    [-20.82, -2.20, 0.16, 0.07],
    [-20.23, -2.47, 0.06, 0.17],
    [-20.32, -2.18, 0.07, 0.12],
    [-20.00, -1.79, 0.04, 0.05],
    [-19.82, -2.24, 0.05, 0.14],
])

labels = [
    (-21.55, -2.35, "GN-z11"),
    (-21.48, -2.51, "GHZ2"),
    (-21.18, -1.94, "Gz9p3"),
    (-20.83, -2.20, "JADES-GS-z14-0"),
    (-20.22, -2.46, "MoM-z14"),
    (-20.30, -2.18, "GHZ8"),
    (-19.98, -1.78, "GHZ1"),
    (-19.80, -2.23, "GHZ7"),
]

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.14)

ax.set_xlim(-22.75, -19.5)
ax.set_ylim(-2.75, -1.50)

ax.grid(True, color=GRID, linestyle=":", linewidth=1.1, alpha=0.75)

for spine in ax.spines.values():
    spine.set_color("#95a3b5")
    spine.set_linewidth(1.1)

ax.tick_params(colors=TEXT, labelsize=13, length=6)

ax.set_xlabel(r"$M_{\mathrm{UV}}$", color=TEXT, fontsize=22, labelpad=14)
ax.set_ylabel(r"$\beta_{\mathrm{UV}}$", color=TEXT, fontsize=22, labelpad=14)

fig.text(
    0.5, 0.945,
    r"BLUE MONSTER CANDIDATES — $\beta_{\mathrm{UV}}$ vs $M_{\mathrm{UV}}$",
    ha="center",
    va="center",
    color="#d7dde8",
    fontsize=23,
    fontweight="bold"
)

fig.text(
    0.5, 0.905,
    "illustrative dark-theme diagnostic diagram • points jitter inside uncertainty ranges",
    ha="center",
    va="center",
    color=MUTED,
    fontsize=13,
    fontweight="bold"
)

# Stellar + nebular limit band
band = ax.axhspan(
    -2.60,
    -2.40,
    color=ORANGE,
    alpha=0.26,
    zorder=0
)

ax.text(
    -22.68,
    -2.385,
    "stellar + nebular limit",
    color=ORANGE,
    fontsize=13,
    fontweight="bold",
    ha="left",
    va="bottom"
)

# Colorbar-like redshift guide
cb_x0, cb_x1 = -21.25, -20.45
cb_y0, cb_y1 = -1.565, -1.52
grad = np.linspace(0, 1, 256).reshape(1, -1)
ax.imshow(
    grad,
    extent=[cb_x0, cb_x1, cb_y0, cb_y1],
    origin="lower",
    aspect="auto",
    cmap="cool",
    alpha=0.95,
    zorder=2
)

ax.text(
    (cb_x0 + cb_x1) / 2,
    -1.505,
    "z",
    color=TEXT,
    fontsize=20,
    fontweight="bold",
    ha="center",
    va="bottom"
)

for tx, label in zip(np.linspace(cb_x0 + 0.08, cb_x1 - 0.08, 4), ["8", "10", "12", "14"]):
    ax.text(
        tx,
        -1.585,
        label,
        color=TEXT,
        fontsize=11,
        ha="center",
        va="top"
    )

# =========================
# Static errorbar containers
# =========================

def draw_errorbars(data, color, alpha, zorder):
    artists = []
    for x, y, ex, ey in data:
        eb = ax.errorbar(
            x,
            y,
            xerr=ex,
            yerr=ey,
            fmt="none",
            ecolor=color,
            elinewidth=1.5,
            capsize=4,
            alpha=alpha,
            zorder=zorder
        )
        artists.append(eb)
    return artists

borg_eb = draw_errorbars(borg, CYAN, 0.70, 4)
bm_eb = draw_errorbars(blue_monsters, MAGENTA, 0.80, 5)

# =========================
# Animated point artists
# =========================

borg_points = []
borg_glows = []

for _ in borg:
    glow = ax.scatter([], [], s=420, marker="*", color=CYAN, alpha=0.0, linewidths=0, zorder=6)
    point = ax.scatter([], [], s=165, marker="*", facecolor=CYAN, edgecolor=BLUE, linewidths=1.2, alpha=0.0, zorder=7)
    borg_glows.append(glow)
    borg_points.append(point)

bm_points = []
bm_glows = []

for _ in blue_monsters:
    glow = ax.scatter([], [], s=380, marker="D", color=MAGENTA, alpha=0.0, linewidths=0, zorder=8)
    point = ax.scatter([], [], s=110, marker="D", facecolor=MAGENTA, edgecolor=WHITE, linewidths=0.8, alpha=0.0, zorder=9)
    bm_glows.append(glow)
    bm_points.append(point)

# Labels
label_artists = []
for x, y, label in labels:
    col = MAGENTA if label in {"JADES-GS-z14-0", "MoM-z14"} else CYAN
    txt = ax.text(
        x + 0.04,
        y + 0.03,
        label,
        color=col,
        fontsize=10,
        fontweight="bold",
        alpha=0.0,
        zorder=10
    )
    label_artists.append(txt)

# HUD
status = ax.text(
    -19.55,
    -2.70,
    "classification layer: idle",
    color=MUTED,
    fontsize=11,
    ha="right",
    va="bottom",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358",
        alpha=0.85
    ),
    zorder=20
)

# =========================
# Animation helpers
# =========================

def smoothstep(edge0, edge1, x):
    if edge0 == edge1:
        return 1.0 if x >= edge1 else 0.0
    t = np.clip((x - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def layer_alpha(t):
    fade_in = smoothstep(0.12, 0.28, t)
    fade_out = 1.0 - smoothstep(0.78, 0.94, t)
    return fade_in * fade_out

def blink_value(t, phase, frequency):
    raw = 0.5 + 0.5 * np.sin(2 * np.pi * (frequency * t + phase))
    return 0.35 + 0.65 * raw**2

def jitter_position(x0, y0, ex, ey, t, phase_x, phase_y):
    # Movement stays inside a fraction of uncertainty range
    dx = 0.30 * ex * np.sin(2 * np.pi * (1.0 * t + phase_x))
    dy = 0.30 * ey * np.sin(2 * np.pi * (1.3 * t + phase_y))
    return x0 + dx, y0 + dy

borg_phases = rng.uniform(0, 1, size=(len(borg), 3))
bm_phases = rng.uniform(0, 1, size=(len(blue_monsters), 3))

# =========================
# Animation update
# =========================

def update(frame):
    t = frame / (FRAMES - 1)
    a = layer_alpha(t)

    # BoRG-JWST points
    for i, ((x0, y0, ex, ey), point, glow) in enumerate(zip(borg, borg_points, borg_glows)):
        px, py = jitter_position(x0, y0, ex, ey, t, borg_phases[i, 0], borg_phases[i, 1])

        blink = blink_value(t, borg_phases[i, 2], 2.0 + 0.25 * (i % 4))
        alpha = a * blink

        point.set_offsets([[px, py]])
        glow.set_offsets([[px, py]])

        point.set_alpha(0.90 * alpha)
        glow.set_alpha(0.16 * alpha)

        point.set_sizes([155 + 35 * blink])
        glow.set_sizes([360 + 140 * blink])

    # Blue monster points: stronger blinking
    for i, ((x0, y0, ex, ey), point, glow) in enumerate(zip(blue_monsters, bm_points, bm_glows)):
        px, py = jitter_position(x0, y0, ex, ey, t, bm_phases[i, 0], bm_phases[i, 1])

        blink = blink_value(t, bm_phases[i, 2], 1.7 + 0.35 * (i % 5))
        pulse = 0.5 + 0.5 * np.sin(2 * np.pi * (6 * t + bm_phases[i, 2]))
        alpha = a * (0.55 + 0.45 * blink)

        point.set_offsets([[px, py]])
        glow.set_offsets([[px, py]])

        point.set_alpha(0.95 * alpha)
        glow.set_alpha((0.18 + 0.20 * pulse) * alpha)

        point.set_sizes([105 + 18 * pulse])
        glow.set_sizes([340 + 190 * pulse])

    # Labels appear after points stabilize
    label_a = a * smoothstep(0.34, 0.48, t)
    for i, txt in enumerate(label_artists):
        local_blink = 0.75 + 0.25 * np.sin(2 * np.pi * (2 * t + i * 0.13))
        txt.set_alpha(label_a * local_blink)

    if a > 0.08:
        status.set_text("classification layer: blue monster candidates active")
        status.set_color(CYAN)
    else:
        status.set_text("classification layer: idle")
        status.set_color(MUTED)

    return (
        borg_points + borg_glows +
        bm_points + bm_glows +
        label_artists +
        [status]
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

# Spectra

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "emission_line_spectrum_dark.gif"

FPS = 24
DURATION_SEC = 8
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#030711"
GRID = "#16324a"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
MAGENTA = "#ff4dff"
ORANGE = "#ff6b32"
WHITE = "#f0f6ff"
GREEN = "#48ffb3"

rng = np.random.default_rng(23)

# =========================
# Synthetic illustrative spectrum
# =========================

x = np.linspace(3.05, 3.85, 900)  # observed wavelength, micron

def gaussian(x, mu, amp, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

continuum = 105 + 18 * np.sin((x - 3.05) * 8.5) + 8 * np.cos((x - 3.05) * 22)

lines = [
    (3.185, 470, 0.006, "[O III]"),
    (3.310, 240, 0.008, "[Ne III]"),
    (3.390, 180, 0.009, "Hε"),
    (3.500, 95, 0.012, "Hδ"),
    (3.705, 420, 0.007, "Hγ + [O III]"),
]

model = continuum.copy()
for mu, amp, sig, _ in lines:
    model += gaussian(x, mu, amp, sig)

noise = rng.normal(0, 38, size=x.size)
obs = model + noise

# Binned/bar-like samples
bin_count = 145
bin_edges = np.linspace(x.min(), x.max(), bin_count + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bin_width = bin_edges[1] - bin_edges[0]

model_binned = np.interp(bin_centers, x, model)
obs_binned = model_binned + rng.normal(0, 52, size=bin_count)
residuals = obs_binned - model_binned

# =========================
# Figure setup
# =========================

fig, (ax, axr) = plt.subplots(
    2, 1,
    figsize=FIGSIZE,
    dpi=DPI,
    sharex=True,
    gridspec_kw={"height_ratios": [3.2, 1.0], "hspace": 0.06}
)

fig.patch.set_facecolor(BG)

for a in (ax, axr):
    a.set_facecolor(BG)
    a.grid(True, color=GRID, linestyle=":", linewidth=1.0, alpha=0.75)
    a.tick_params(colors=TEXT, labelsize=12, length=5)

    for spine in a.spines.values():
        spine.set_color("#95a3b5")
        spine.set_linewidth(1.1)

ax.set_xlim(3.05, 3.85)
ax.set_ylim(20, 640)
axr.set_ylim(-140, 140)

ax.set_ylabel("Flux", color=TEXT, fontsize=17, labelpad=12)
axr.set_ylabel("Residual", color=TEXT, fontsize=14, labelpad=10)
axr.set_xlabel(r"$\lambda_{\mathrm{obs}}$  ($\mu$m)", color=TEXT, fontsize=18, labelpad=12)

fig.text(
    0.5, 0.945,
    "JWST / NIRSpec-LIKE EMISSION LINE SPECTRUM",
    ha="center",
    va="center",
    color="#d7dde8",
    fontsize=23,
    fontweight="bold"
)

fig.text(
    0.5, 0.905,
    "illustrative observed-frame spectrum • line model + residuals",
    ha="center",
    va="center",
    color=MUTED,
    fontsize=13,
    fontweight="bold"
)

# =========================
# Static line markers
# =========================

for mu, amp, sig, label in lines:
    ax.axvline(mu, color=CYAN, linestyle=":", linewidth=1.0, alpha=0.38)
    axr.axvline(mu, color=CYAN, linestyle=":", linewidth=1.0, alpha=0.25)

    ax.text(
        mu,
        628,
        label,
        color=CYAN,
        fontsize=10,
        fontweight="bold",
        ha="center",
        va="top",
        rotation=90,
        alpha=0.85
    )

# S/N diagnostic box
sn_text = (
    "S/N:\n"
    "[O III] = 10.9\n"
    "[Ne III] = 9.2\n"
    "Hε = 9.0\n"
    "Hδ = 4.8\n"
    r"$H_\gamma$ + [O III] = 6.6"
)

ax.text(
    3.61,
    575,
    sn_text,
    color=ORANGE,
    fontsize=11,
    ha="left",
    va="top",
    bbox=dict(
        boxstyle="round,pad=0.38",
        facecolor="#07111f",
        edgecolor=ORANGE,
        alpha=0.82
    ),
    zorder=30
)

axr.text(
    3.055,
    -125,
    r"$\chi^2_\nu = 2.09$",
    color=MUTED,
    fontsize=11,
    ha="left",
    va="bottom"
)

# =========================
# Animated artists
# =========================

# Upper panel: observed histogram-like spectrum
bar_artists = ax.bar(
    bin_centers,
    np.zeros_like(bin_centers),
    width=bin_width * 0.86,
    bottom=0,
    color=MAGENTA,
    edgecolor=MAGENTA,
    alpha=0.23,
    linewidth=0.4,
    zorder=5
)

# Lower residual bars
res_bar_artists = axr.bar(
    bin_centers,
    np.zeros_like(bin_centers),
    width=bin_width * 0.86,
    bottom=0,
    color=MAGENTA,
    edgecolor=MAGENTA,
    alpha=0.30,
    linewidth=0.4,
    zorder=5
)

# Model curve
glow1, = ax.plot([], [], color=ORANGE, linewidth=8, alpha=0.10, zorder=12)
glow2, = ax.plot([], [], color=ORANGE, linewidth=4, alpha=0.18, zorder=13)
model_line, = ax.plot([], [], color=ORANGE, linewidth=2.2, zorder=14)

# Residual model zero line
axr.axhline(0, color=TEXT, linewidth=0.9, alpha=0.35, zorder=3)
res_line, = axr.plot([], [], color=CYAN, linewidth=1.6, alpha=0.8, zorder=12)

# Peak highlight markers
peak_glows = []
peak_points = []
peak_labels = []

for mu, amp, sig, label in lines:
    y_peak = np.interp(mu, x, model)

    glow = ax.scatter([], [], s=560, color=CYAN, alpha=0.0, linewidths=0, zorder=20)
    point = ax.scatter([], [], s=80, color=WHITE, edgecolor=CYAN, linewidths=1.0, alpha=0.0, zorder=21)

    txt = ax.text(
        mu + 0.01,
        min(y_peak + 45, 600),
        label,
        color=CYAN,
        fontsize=10,
        fontweight="bold",
        alpha=0.0,
        zorder=22
    )

    peak_glows.append(glow)
    peak_points.append(point)
    peak_labels.append(txt)

cursor, = ax.plot([], [], marker="o", markersize=6, color=WHITE, alpha=0.95, zorder=25)

status = ax.text(
    3.84,
    38,
    "line scan: idle",
    color=MUTED,
    fontsize=11,
    ha="right",
    va="bottom",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358",
        alpha=0.85
    ),
    zorder=30
)

# =========================
# Animation helpers
# =========================

def smoothstep(edge0, edge1, value):
    t = np.clip((value - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def ease(t):
    return 1 - (1 - t) ** 3

def layer_alpha(t):
    fade_in = smoothstep(0.05, 0.18, t)
    fade_out = 1.0 - smoothstep(0.88, 0.98, t)
    return fade_in * fade_out

# =========================
# Animation update
# =========================

def update(frame):
    t = frame / (FRAMES - 1)
    a = layer_alpha(t)

    scan = ease(t)
    xmax = x.min() + scan * (x.max() - x.min())

    visible_line = x <= xmax
    visible_bins = bin_centers <= xmax

    xv = x[visible_line]
    yv = model[visible_line]

    model_line.set_data(xv, yv)
    glow1.set_data(xv, yv)
    glow2.set_data(xv, yv)

    if len(xv) > 0:
        cursor.set_data([xv[-1]], [yv[-1]])

    for i, bar in enumerate(bar_artists):
        if visible_bins[i]:
            bar.set_height(obs_binned[i])
            bar.set_alpha(0.20 + 0.15 * a)
        else:
            bar.set_height(0)
            bar.set_alpha(0)

    for i, bar in enumerate(res_bar_artists):
        if visible_bins[i]:
            h = residuals[i]
            bar.set_y(0 if h >= 0 else h)
            bar.set_height(abs(h))
            bar.set_alpha(0.26 + 0.16 * a)
        else:
            bar.set_height(0)
            bar.set_alpha(0)

    res_visible = bin_centers <= xmax
    res_line.set_data(bin_centers[res_visible], residuals[res_visible])

    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * 5 * t)

    for i, ((mu, amp, sig, label), glow, point, txt) in enumerate(
        zip(lines, peak_glows, peak_points, peak_labels)
    ):
        passed = smoothstep(mu - 0.015, mu + 0.025, xmax)
        local_alpha = a * passed

        y_peak = np.interp(mu, x, model)

        glow.set_offsets([[mu, y_peak]])
        point.set_offsets([[mu, y_peak]])

        glow.set_alpha((0.10 + 0.18 * pulse) * local_alpha)
        point.set_alpha(0.95 * local_alpha)
        txt.set_alpha(0.88 * local_alpha)

        glow.set_sizes([420 + 260 * pulse])

    if a > 0.08:
        status.set_text("line scan: emission features detected")
        status.set_color(CYAN)
    else:
        status.set_text("line scan: idle")
        status.set_color(MUTED)

    return (
        list(bar_artists)
        + list(res_bar_artists)
        + [model_line, glow1, glow2, res_line, cursor, status]
        + peak_glows
        + peak_points
        + peak_labels
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")